In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv('^SPX.csv')

log_returns = pd.DataFrame({'log_returns': np.log(df['Close'].shift(-1) / df['Close'])})

log_returns = log_returns.dropna().reset_index(drop=True)

description = log_returns['log_returns'].describe()

num_iterations = 5
bins = [log_returns['log_returns'].quantile((i+1)/(num_iterations)) for i in range(num_iterations-1)]
observations = [log_returns['log_returns'].quantile((i+1)/(num_iterations+1)) for i in range(num_iterations)]

In [14]:
from arch import arch_model

model = arch_model(log_returns, vol='Garch', p=2, q=2)
res = model.fit()

sim_data = res.model.simulate(
    res.params,
    nobs=1000
)

generated_returns = sim_data['data']
generated_volatility = sim_data['volatility']

Iteration:      1,   Func. Count:      8,   Neg. LLF: 1.6828000176247737e+23
Iteration:      2,   Func. Count:     24,   Neg. LLF: -28874.40684230255
Optimization terminated successfully    (Exit mode 0)
            Current function value: -28874.40682123681
            Iterations: 6
            Function evaluations: 24
            Gradient evaluations: 2


c:\dev\environments\QHMM_MLE\Lib\site-packages\arch\univariate\base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0001294. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)


In [15]:
res.params

mu          0.000619
omega       0.000003
alpha[1]    0.100000
alpha[2]    0.100000
beta[1]     0.390000
beta[2]     0.390000
Name: params, dtype: float64

In [ ]:
'''
Copyright 2025 Jack Morgan

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
'''

from abc import ABC, abstractmethod

class HMM(ABC):
    @abstractmethod
    def log_likelihood(self, sequence):
        pass
    @abstractmethod
    def generate_sequence(self, length):
        pass

Constant Mean(constant: yes, no. of exog: 0, volatility: GARCH(p: 1, q: 1), distribution: Normal distribution), id: 0x1f0d8a0a550

In [ ]:
class GARCH_Model(HMM):
    def __init__(self, p, q):
        self.p = p
        self.q = q
        self.model = arch_model(log_returns, vol='Garch', p=p, q=q)
        self.res = self.model.fit(disp='off')
    
    def log_likelihood(self, sequence):
        sim_data = self.res.model.simulate(
            self.res.params,
            nobs=len(sequence)
        )
        generated_returns = sim_data['data']
        return np.sum(np.log(generated_returns))
    
    def generate_sequence(self, length):
        sim_data = self.res.model.simulate(
            self.res.params,
            nobs=length
        )
        return sim_data['data']

0     -0.182376
1     -0.189300
2     -0.177184
3     -0.160593
4     -0.199422
         ...   
995   -0.190728
996   -0.185012
997   -0.197245
998   -0.173511
999   -0.191214
Name: data, Length: 1000, dtype: float64